## Fraud Detection ML Pipeline

This section implements a fraud detection model using **Logistic Regression** (supervised binary classification) trained on labeled transactions.

### Pipeline Steps:

#### 1. Feature Engineering (SQL)
Creates `fraud_detection_features` table with:
* **Temporal Features**: year, month, day, hour, day_of_week, is_weekend, is_night
* **Card Features**: card_brand, card_type, credit_limit, has_chip, card_on_dark_web, amount_to_limit_ratio
* **Behavioral Features**: 
  * User-level: avg/stddev/min/max amounts, total transactions
  * Card-level: avg amount, transaction count
  * Monthly aggregations: total amount, transaction count
* **Anomaly Indicators**: z_score_amount, high_limit_usage, is_amount_outlier

#### 2. ML Preparation (Python)
* Loads feature table and converts to pandas
* Encodes categorical variables (card_brand, card_type)
* Handles missing values with median imputation
* Scales features using StandardScaler
* Total: **27 numeric features**

#### 3. Model Training (Logistic Regression)
* Supervised binary classification on the labeled feature set
* `class_weight='balanced'` to handle the 0.15% fraud rate
* Preprocessing (imputation, scaling, one-hot encoding) wrapped in an sklearn Pipeline
* Tracked in MLflow for reproducibility

#### 4. Out-of-Sample Evaluation
* Held-out 20% stratified test set
* Confusion matrix, accuracy, precision, recall
* Confusion matrix heatmap visualization

#### 5. Output
* Evaluation results saved to `payments.default.fraud_lr_evaluation_results`:
  * Metrics: accuracy, precision, recall
  * Confusion matrix components (TN, FP, FN, TP)

### Usage:
Run cells sequentially to train the model. Review evaluation metrics in the `fraud_lr_evaluation_results` table.

In [0]:
%sql
-- Comprehensive fraud detection feature set for supervised learning
CREATE OR REPLACE TABLE payments.default.fraud_detection_features AS
WITH transaction_features AS (
  SELECT 
    trx.*,
    -- Temporal features
    year(trx.date) as year,
    month(trx.date) as month,
    dayofmonth(trx.date) as day,
    dayofweek(trx.date) as day_of_week,
    hour(trx.date) as hour,
    CASE WHEN dayofweek(trx.date) IN (1,7) THEN 1 ELSE 0 END as is_weekend,
    CASE WHEN hour(trx.date) >= 22 OR hour(trx.date) <= 6 THEN 1 ELSE 0 END as is_night,
    
    -- Card features
    crd.card_brand,
    crd.card_type,
    CAST(REPLACE(crd.credit_limit, '$', '') AS DOUBLE) as credit_limit,
    crd.has_chip,
    crd.num_cards_issued,
    CASE WHEN crd.card_on_dark_web = 'Yes' THEN 1 ELSE 0 END as card_on_dark_web,
    
    -- User demographic features
    usr.current_age,
    usr.gender,
    usr.credit_score,
    CAST(REPLACE(usr.yearly_income, '$', '') AS DOUBLE) as yearly_income,
    CAST(REPLACE(usr.total_debt, '$', '') AS DOUBLE) as total_debt,
    usr.num_credit_cards,
    
    -- Amount to limit ratio
    trx.amount / NULLIF(CAST(REPLACE(crd.credit_limit, '$', '') AS DOUBLE), 0) as amount_to_limit_ratio,
    
    -- User aggregated features (behavioral patterns)
    AVG(trx.amount) OVER (PARTITION BY trx.client_id) as user_avg_amount,
    STDDEV(trx.amount) OVER (PARTITION BY trx.client_id) as user_stddev_amount,
    MIN(trx.amount) OVER (PARTITION BY trx.client_id) as user_min_amount,
    MAX(trx.amount) OVER (PARTITION BY trx.client_id) as user_max_amount,
    COUNT(*) OVER (PARTITION BY trx.client_id) as user_total_transactions,
    
    -- Card-level aggregations
    AVG(trx.amount) OVER (PARTITION BY trx.card_id) as card_avg_amount,
    COUNT(*) OVER (PARTITION BY trx.card_id) as card_total_transactions,
    
    -- Time-based aggregations (monthly)
    SUM(trx.amount) OVER (PARTITION BY trx.client_id, year(trx.date), month(trx.date)) as monthly_total_amount,
    COUNT(*) OVER (PARTITION BY trx.client_id, year(trx.date), month(trx.date)) as monthly_transaction_count,
    
    -- Deviation from user's normal behavior
    (trx.amount - AVG(trx.amount) OVER (PARTITION BY trx.client_id)) / 
      NULLIF(STDDEV(trx.amount) OVER (PARTITION BY trx.client_id), 0) as z_score_amount,
    
    -- Fraud label (target)
    CASE WHEN lbl.is_fraud = 'Yes' THEN 1 ELSE 0 END as is_fraud
    
  FROM payments.default.pay_transactions trx
  LEFT JOIN payments.default.pay_cards crd ON trx.card_id = crd.id
  LEFT JOIN payments.default.pay_users usr ON trx.client_id = usr.id
  INNER JOIN payments.default.pay_labels lbl ON trx.id = lbl.transaction_id
)
SELECT 
  *
FROM transaction_features;

In [0]:
# ============================================================
# Step 1: EDA — Load feature table & check class distribution
# ============================================================
df_spark = spark.table("payments.default.fraud_detection_features")
total_rows = df_spark.count()
print(f"Total rows: {total_rows:,}")

# Class distribution
print("\n=== Class Distribution ===")
class_dist = df_spark.groupBy("is_fraud").count().orderBy("is_fraud")
display(class_dist)

fraud_count = df_spark.filter("is_fraud = 1").count()
print(f"\nFraud:  {fraud_count:,} ({fraud_count/total_rows*100:.2f}%)")
print(f"Normal: {total_rows - fraud_count:,} ({(total_rows - fraud_count)/total_rows*100:.2f}%)")

In [0]:
# ============================================================
# Step 2: Data Preparation & Train/Test Split
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split

# Convert to pandas
df = df_spark.toPandas()
print(f"DataFrame shape: {df.shape}")

# Define target and columns to exclude (IDs, raw date, high-cardinality)
target = "is_fraud"
exclude_cols = [target, "id", "date", "client_id", "card_id",
                "merchant_id", "merchant_city", "zip"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].copy()
y = df[target].astype(int).copy()

# Explicitly define categorical vs numeric columns
categorical_cols = ["use_chip", "is_online", "merchant_state",
                    "card_brand", "card_type", "has_chip",
                    "gender", "errors"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# Convert any Decimal-typed columns to float (Spark DECIMAL → pandas)
for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce").astype(float)

print(f"\nNumeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Stratified train/test split (fraud is <1% of data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\nTrain: {len(X_train):,} rows ({y_train.sum():,} fraud, {y_train.mean()*100:.2f}%)")
print(f"Test:  {len(X_test):,} rows ({y_test.sum():,} fraud, {y_test.mean()*100:.2f}%)")

In [0]:
# ============================================================
# Step 3: Train Logistic Regression (class-weighted)
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Build preprocessing pipeline for numeric and categorical features
# Fraud is only 0.15% of rows — class_weight='balanced' compensates for imbalance
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),  # Fill missing numerics with median
            ("scaler", StandardScaler()),                   # Standardize to mean=0, std=1
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),  # Fill missing categoricals
            ("onehot", OneHotEncoder(handle_unknown="ignore")),  # Convert to binary columns
        ]), categorical_cols),
    ]
)

# Combine preprocessing + model into a single pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",  # Penalize misclassifying minority class (fraud) more heavily
        solver="lbfgs",           # Optimization algorithm (works well for smaller datasets)
        C=1.0,                    # Regularization strength (inverse; higher = less regularization)
        max_iter=1000,            # Max iterations for convergence
        random_state=42,          # Reproducible results
    )),
])

# Train model and track in MLflow for reproducibility and comparison
with mlflow.start_run(run_name="fraud_logistic_regression"):
    # Log hyperparameters
    mlflow.log_params({
        "model_type": "logistic_regression",
        "class_weight": "balanced",
        "C": 1.0,
        "max_iter": 1000,
        "n_train_rows": len(X_train),
        "n_test_rows": len(X_test),
    })

    # Fit the model on training data (1,355,965 rows)
    model.fit(X_train, y_train)

    # Generate predictions on the held-out test set (338,992 rows)
    y_pred = model.predict(X_test)          # Binary predictions (0 or 1)
    y_prob = model.predict_proba(X_test)[:, 1]  # Fraud probability scores (0 to 1)

    # Log all evaluation metrics to MLflow
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, y_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
    mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, y_prob))

    # Log the trained model with example input for serving
    input_example = X_test.head(5)
    signature = infer_signature(input_example, model.predict(input_example))
    mlflow.sklearn.log_model(model, "model", signature=signature, input_example=input_example)

print("Training complete — logistic regression fitted, test-set predictions generated.")

In [0]:
# ============================================================
# Step 4: Out-of-Sample Evaluation — Confusion Matrix & Metrics
# ============================================================
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from datetime import datetime, timezone

# Compute confusion matrix from test set predictions
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()  # Extract all 4 components

# Calculate evaluation metrics on out-of-sample test set (338,992 transactions)
accuracy = accuracy_score(y_test, y_pred)      # Overall correctness
precision = precision_score(y_test, y_pred)    # How many flagged are truly fraud?
recall = recall_score(y_test, y_pred)          # How many frauds did we catch?
f1 = f1_score(y_test, y_pred)                  # Harmonic mean of precision & recall

print("=" * 58)
print("OUT-OF-SAMPLE EVALUATION — LOGISTIC REGRESSION (test set)")
print("=" * 58)
print(f"Confusion Matrix:  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print(f"Accuracy:  {accuracy:.4f}  (overall correctness across both classes)")
print(f"Precision: {precision:.4f}  (of flagged transactions, % truly fraud)")
print(f"Recall:    {recall:.4f}  (of all frauds, % successfully caught)")
print(f"F1 Score:  {f1:.4f}  (harmonic mean balancing precision & recall)")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Not Fraud", "Fraud"]).plot(
    ax=ax, cmap="Blues", values_format=","
)
ax.set_title("Confusion Matrix — Logistic Regression (Out-of-Sample)")
plt.tight_layout()
plt.show()

# ============================================================
# Step 5: Save evaluation results to Unity Catalog
# ============================================================
# Package all metrics and confusion matrix components into a single row
results_df = pd.DataFrame([{
    "model": "logistic_regression",
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "tn": int(tn),
    "fp": int(fp),
    "fn": int(fn),
    "tp": int(tp),
    "test_rows": int(len(y_test)),
    "evaluated_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
}])

# Write results to Unity Catalog table for tracking and comparison
(spark.createDataFrame(results_df)
      .write.mode("overwrite")  # Replace any existing results
      .option("overwriteSchema", "true")  # Allow schema changes
      .saveAsTable("payments.default.fraud_lr_evaluation_results"))

print("\nResults saved to payments.default.fraud_lr_evaluation_results")
display(spark.table("payments.default.fraud_lr_evaluation_results"))

### Results Interpretation

#### Confusion Matrix Breakdown
* **True Negatives (TN = 324,141):** Normal transactions correctly identified as legitimate. This represents 95.6% of all test transactions.
* **False Positives (FP = 14,329):** Normal transactions incorrectly flagged as fraud. About 4.2% of legitimate transactions trigger false alarms.
* **False Negatives (FN = 3):** Fraudulent transactions that slipped through undetected. Only 3 frauds missed out of 522 total.
* **True Positives (TP = 519):** Fraudulent transactions successfully caught by the model.

#### Metric Analysis

**Accuracy (95.77%)** — High overall correctness, but misleading in severely imbalanced datasets. Since 99.85% of transactions are legitimate, a naive "always predict not fraud" model would achieve 99.85% accuracy. Our model performs slightly worse on overall accuracy because it aggressively flags fraud.

**Precision (3.50%)** — Of all transactions flagged as fraud, only 3.5% are actually fraudulent. This means **96.5% of fraud alerts are false positives**. For every genuine fraud caught, the model incorrectly flags ~27 legitimate transactions. This is the cost of aggressive fraud detection with extreme class imbalance (0.15% fraud rate).

**Recall (99.43%)** — The model catches **99.43% of all fraud cases**, missing only 3 out of 522 fraudulent transactions. This is the model's primary strength — it's extremely sensitive to fraud, which is critical in fraud detection where missing fraud is costlier than false alarms.

**F1 Score (6.75%)** — The harmonic mean of precision and recall. The low F1 reflects the severe imbalance: excellent recall but poor precision. This is expected and acceptable in fraud detection where the priority is catching fraud, even at the cost of false positives.

#### Business Implications

**✓ Strengths:**
* Catches virtually all fraud (99.43% recall) — only 3 frauds slip through
* `class_weight='balanced'` successfully compensates for the 0.15% fraud rate
* Low false negative rate minimizes financial losses from undetected fraud

